# Customer Churn & Revenue Intelligence

**Business problem:** A telecom subscription business loses about one in three customers. Where is the churn coming from, how much revenue does it cost, and which customers should the retention team call first?

**Data:** 2,000 customers (city, age, contract, tenure, monthly/total charges, internet service, signup date, churn status).

**What this notebook does**
1. Cleans and checks the data
2. Answers 10 business questions on churn and revenue (Q1-Q10)
3. Tests which factors are statistically significant
4. Builds a churn prediction model and scores every active customer
5. Estimates the revenue that retention actions could save, and gives recommendations

**Tools:** Python, pandas, matplotlib/seaborn, SciPy, scikit-learn.

**Churn rate definition:** Yes / (Yes + No). The 35 customers with unknown churn status are excluded.

## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

# 1. Data load
FILE = r"C:\Users\HP\Downloads\clean_customer_data_2000.xlsx"
df = pd.read_excel(FILE, sheet_name="Clean_Data")

# 2. Junk columns hatao + column names saaf karo
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)
df = df.rename(columns={
    "Tenure (Months)": "Tenure", "Monthly Charges (₹)": "MonthlyCharges",
    "Total Charges (₹)": "TotalCharges", "Tenure Bucket": "TenureBucket",
    "Contract Type": "Contract", "Internet Service": "Internet",
})

# 3. AgeGroup banao
age_order = ["18-25", "26-35", "36-45", "46-60", "60+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=[17, 25, 35, 45, 60, 200], labels=age_order)

# 4. Known churn wale customers
known = df[df["Churn"].isin(["Yes", "No"])].copy()
known["ChurnFlag"] = (known["Churn"] == "Yes").astype(int)
churned_df = known[known["ChurnFlag"] == 1]

# 5. Helper function
def churn_by(col, data=None, order=None):
    data = known if data is None else data
    t = (data.groupby(col, observed=True)
             .agg(customers=("CustID", "count"), churned=("ChurnFlag", "sum"),
                  churn_rate=("ChurnFlag", "mean"), avg_monthly_charge=("MonthlyCharges", "mean"))
             .assign(churn_rate=lambda x: x["churn_rate"] * 100))
    return t.reindex(order) if order else t

print(df.shape, "| known:", len(known), "| churned:", len(churned_df))

## 2. Data quality check

In [ ]:
print("Rows:", len(df), "| Unique customers:", df["CustID"].nunique(), "| Duplicates:", df.duplicated("CustID").sum())
print("\nMissing values:")
print(df.isna().sum()[lambda s: s > 0])
print("\nChurn status:")
print(df["Churn"].value_counts())
print("\nRows flagged for review:", (df["Needs_Review"] != "No").sum())

## 3. Business questions

### Q1. What are the headline KPIs?

In [ ]:
total = len(df)
churned = (known["ChurnFlag"] == 1).sum()
rate = known["ChurnFlag"].mean() * 100

churned_df = known[known["ChurnFlag"] == 1]
monthly_rev_total = known["MonthlyCharges"].sum()
monthly_rev_lost = churned_df["MonthlyCharges"].sum()

kpis = pd.Series({
    "Total customers": total,
    "Churned customers": churned,
    "Churn rate (%)": rate,
    "Monthly revenue of known customers (₹)": monthly_rev_total,
    "Monthly revenue lost to churn (₹)": monthly_rev_lost,
    "Share of monthly revenue lost (%)": monthly_rev_lost / monthly_rev_total * 100,
    "Annualised revenue lost (₹)": monthly_rev_lost * 12,
    "Avg monthly charge - churned (₹)": churned_df["MonthlyCharges"].mean(),
    "Avg monthly charge - retained (₹)": known.loc[known["ChurnFlag"] == 0, "MonthlyCharges"].mean(),
})
kpis.to_frame("Value")

**Business insight**
- 665 of the 1,965 customers with a known status have churned, a **churn rate of 33.8%**. One in three customers is leaving, which is very high.
- Churned customers represent **₹5.67 lakh of monthly revenue (34.5% of the ₹16.46 lakh base)**, about **₹68 lakh per year**.
- Churned customers pay slightly more (₹866 vs ₹837 a month), so the loss in revenue is a bit larger than the loss in customers.

### Q2. Which contract type churns the most?

In [ ]:
contract = churn_by("Contract").drop("Unknown", errors="ignore")
display(contract)
share = (known[known.ChurnFlag == 1]["Contract"].value_counts(normalize=True) * 100).round(1)
print("Share of all churn by contract (%):\n", share)

contract["churn_rate"].plot(kind="bar", color="#c0392b", figsize=(6, 4), title="Churn rate by contract (%)")
plt.ylabel("%"); plt.xticks(rotation=0); plt.show()

**Business insight**
- Contract type is the **strongest driver of churn**: Month-to-month 47.8%, One year 20.2%, Two year 10.9%.
- Month-to-month customers are 55% of the base but give **~78% of all churn**.
- Average monthly charge is almost identical across contracts (~₹845), so customers leave because there is no commitment, not because of price.
- **Action:** target Month-to-month customers first with incentives to move to a 1- or 2-year plan.

### Q3. Does churn fall as customers stay longer (tenure)?

In [ ]:
order = ["0-6 months", "7-12 months", "13-24 months", "25-48 months", "49+ months"]
tenure = churn_by("TenureBucket", order=order)
display(tenure)
tenure["churn_rate"].plot(kind="bar", color="#2c7fb8", figsize=(7, 4), title="Churn rate by tenure bucket (%)")
plt.ylabel("%"); plt.xticks(rotation=0); plt.show()

**Business insight**
- Churn falls steadily with tenure: **49.7%** in the first 6 months, 38.2% at 7-12 months, and only **8.6%** after 49 months.
- About **51% of all churn (337 of 665) happens in the first 12 months**.
- The big drop comes after month 24 (35.0% to 16.6%): customers who stay past two years usually stay.
- **Action:** invest in onboarding and early check-ins during the first year.

### Q4. Which age groups are most likely to leave?

In [ ]:
age_order = ["18-25", "26-35", "36-45", "46-60", "60+"]
age = churn_by("AgeGroup", order=age_order)
display(age)
age["churn_rate"].plot(kind="bar", color="#8e44ad", figsize=(6, 4), title="Churn rate by age group (%)")
plt.ylabel("%"); plt.xticks(rotation=0); plt.show()

**Business insight**
- Churn is fairly even across age groups (30.3% to 37.2%). The highest is **36-45 (37.2%)** and the lowest is **26-35 (30.3%)**.
- Age is a **weak driver** compared with contract and tenure.
- The 46-60 and 60+ groups lose the most revenue (about 27% and 26% of the total) mainly because they are the largest groups, not because they churn more.

### Q5. How does internet service type affect churn?

In [ ]:
internet = churn_by("Internet").drop("Unknown", errors="ignore")
display(internet)

**Business insight**
- **Fiber customers churn the most (37.7%)**, ahead of No Internet (35.1%) and DSL (28.7%).
- Fiber customers pay about ₹1,090 a month versus ₹754 for DSL, so each lost Fiber customer costs more.
- **Action:** review Fiber service quality and pricing; it is the most valuable segment and also the leakiest.

### Q6. Which cities have the highest churn?

In [ ]:
city = churn_by("City").drop("Unknown", errors="ignore").sort_values("churn_rate", ascending=False)
display(city)
city["churn_rate"].plot(kind="barh", figsize=(7, 5), color="#e67e22", title="Churn rate by city (%)").invert_yaxis()
plt.show()

**Business insight**
- City matters little: churn ranges from **28.8% (Kolkata) to 38.1% (Chennai)**, a spread of about 9 points.
- Chennai, Indore and Chandigarh are the highest-churn cities; Kolkata, Bangalore and Ahmedabad the lowest.
- Because every city has 120-150 customers, small differences may be noise. Treat city as a secondary factor.

### Q7. Where is the revenue loss concentrated?

In [ ]:
def revenue_lost_by(col):
    t = (churned_df.groupby(col)["MonthlyCharges"].agg(customers_lost="count", monthly_revenue_lost="sum")
         .sort_values("monthly_revenue_lost", ascending=False))
    t["% of total loss"] = t["monthly_revenue_lost"] / t["monthly_revenue_lost"].sum() * 100
    return t

for col in ["Contract", "Internet", "AgeGroup", "City"]:
    print(f"--- Monthly revenue lost by {col} ---")
    display(revenue_lost_by(col).head(6))

**Business insight**
- **Month-to-month contracts account for ~77% of lost monthly revenue** and **Fiber for ~63%**.
- Fixing these two segments covers most of the revenue problem.
- Delhi has the largest city-level loss (8.3% of the total), but the city shares are all close to each other.

### Q8. Which combination of contract and internet service is riskiest?

In [ ]:
pivot = known[(known.Contract != "Unknown") & (known.Internet != "Unknown")].pivot_table(
    index="Contract", columns="Internet", values="ChurnFlag", aggfunc="mean") * 100
plt.figure(figsize=(6, 3.5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Reds")
plt.title("Churn rate (%) - Contract x Internet service"); plt.show()

**Business insight**
- The riskiest groups are **Month-to-month with No Internet (52.4%)** and **Month-to-month with Fiber (50.4%)**: roughly one in two leaves.
- The safest group is **Two year with No Internet (4.7%)**.
- Even on a 1-year contract, Fiber customers churn more (26.3%) than DSL customers (15.7%), which supports a Fiber service review.

### Q9. Has churn changed across signup-year cohorts?

In [ ]:
known["SignupYear"] = pd.to_datetime(known["Signup Date"]).dt.year
cohort = churn_by("SignupYear", data=known.dropna(subset=["SignupYear"]))
cohort.index = cohort.index.astype(int)
display(cohort)

**Business insight**
- Churn rises sharply for newer signups: from **3.6-11.5% for 2019-2021 cohorts** to **36.8% (2024) and 46.1% (2025)**.
- Part of this is natural, since new customers are still in the risky first year (see Q3). But the gap is large enough to also suggest weaker customer quality or onboarding in recent cohorts.
- **Action:** compare the acquisition channels and offers used in 2024-2025 with earlier years.

### Q10. Do churned customers pay more per month?

In [ ]:
print(known.groupby("Churn")["MonthlyCharges"].describe())
sns.boxplot(data=known, x="Churn", y="MonthlyCharges", palette=["#27ae60", "#c0392b"], hue="Churn", legend=False)
plt.title("Monthly charges: retained vs churned (₹)"); plt.show()

**Business insight**
- Churned customers pay slightly more: **median ₹899 vs ₹849** (mean ₹866 vs ₹837).
- Price plays a role, but a minor one. The spread is wide for both groups, so price alone does not explain who leaves.

## 4. Are these differences real? Statistical tests
The tables above show differences, but with samples of 100-500 per group some could be chance. A chi-square test checks whether each factor is really linked to churn, and Cramer's V shows how strong the link is (0 = none, 1 = perfect).

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(ct):
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

rows = []
for col in ["Contract", "TenureBucket", "Internet", "AgeGroup", "City", "Gender"]:
    d = known[known[col].astype(str) != "Unknown"]
    ct = pd.crosstab(d[col], d["ChurnFlag"])
    chi2, p, dof, _ = chi2_contingency(ct)
    rows.append({"Feature": col, "Chi-square": chi2, "p-value": p,
                 "Cramer's V (effect size)": cramers_v(ct),
                 "Significant (p<0.05)": "Yes" if p < 0.05 else "No"})

stats_df = pd.DataFrame(rows).sort_values("Cramer's V (effect size)", ascending=False).reset_index(drop=True)
stats_df

**Business insight**
- **Contract type (V = 0.34), tenure (V = 0.25) and internet service (V = 0.09) are statistically significant** (p < 0.05). Contract type is by far the strongest link.
- **City (p = 0.88), age group (p = 0.39) and gender (p = 0.46) are not significant.** The small differences seen in Q4 and Q6 are consistent with random variation.
- This tells the business where to spend: retention offers should be designed around contract and tenure, not around age or city.

## 5. Predicting churn
A model that gives each customer a churn probability lets the retention team prioritise. I use only the features that were significant above (tenure, monthly charges, contract, internet service) and compare Logistic Regression with Random Forest.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay

# Only the features that were statistically significant in the chi-square tests (Age, City, Gender were not)
num_cols = ["Tenure", "MonthlyCharges"]
cat_cols = ["Contract", "Internet"]

model_df = known.copy()
for c in cat_cols:                       # treat "Unknown" as missing so it is imputed
    model_df[c] = model_df[c].replace("Unknown", np.nan)

X = model_df[num_cols + cat_cols]
y = model_df["ChurnFlag"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

def make_pipeline(model):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])
    return Pipeline([("pre", pre), ("model", model)])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                                          class_weight="balanced", random_state=42)),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = {"ROC-AUC": roc_auc_score(y_test, proba),
                     "Accuracy": (pipe.predict(X_test) == y_test).mean()}
pd.DataFrame(results).T

In [ ]:
best_name = max(results, key=lambda k: results[k]["ROC-AUC"])
best = models[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best.predict(X_test), target_names=["Retained", "Churned"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, display_labels=["Retained", "Churned"], cmap="Blues", ax=ax[0])
ax[0].set_title("Confusion matrix (test set)")
for name, pipe in models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax[1])
ax[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
ax[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

**Business insight**
- Both models reach a **ROC-AUC of about 0.75** on the held-out test set (about 0.71 in cross-validation), a **moderate but useful** ability to rank customers by risk. A random ranking would score 0.50.
- The model **catches 84% of the customers who actually churn** (recall), at the cost of precision (51%): about half of the customers it flags will not leave. For a retention campaign, where a wasted offer is cheaper than a lost customer, this is an acceptable trade-off.
- The model is limited by the data: it has only four usable features. Adding usage, complaints or payment history would improve it.

In [ ]:
# Which factors drive churn? (Logistic Regression coefficients on standardised / one-hot features)
lr = models["Logistic Regression"]
names = lr.named_steps["pre"].get_feature_names_out()
coef = pd.Series(lr.named_steps["model"].coef_[0], index=names).sort_values()
coef.index = coef.index.str.replace("num__", "").str.replace("cat__", "")

coef.plot(kind="barh", figsize=(7, 4), color=np.where(coef > 0, "#c0392b", "#27ae60"),
         title="Churn drivers (red = raises churn risk, green = lowers it)")
plt.axvline(0, color="black", lw=0.8); plt.xlabel("Logistic regression coefficient"); plt.show()

**Business insight**
- **Month-to-month contract is the biggest risk factor** (+0.84), and a **two-year contract is the biggest protective factor** (-0.62).
- **Longer tenure lowers risk** (-0.27) and **higher monthly charges raise it slightly** (+0.13).
- Internet service matters much less once contract and tenure are taken into account. This is why the Fiber churn seen earlier is largely a Fiber *and* Month-to-month effect (see Q8).

## 6. Who should the retention team call first?
Every customer who is still active gets an out-of-fold churn probability (so no customer is scored by a model that has seen their own outcome), and is placed in a Low (<25%), Medium (25-50%) or High (>50%) risk band.

In [ ]:
# Out-of-fold churn probability for every customer with a known status (no data leakage)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# (no class_weight here, so the probabilities stay realistic for revenue calculations)
oof = cross_val_predict(make_pipeline(LogisticRegression(max_iter=1000)),
                        X, y, cv=cv, method="predict_proba")[:, 1]
model_df["ChurnProb"] = oof

# Focus on customers who are STILL active
active = model_df[model_df["Churn"] == "No"].copy()
active["RiskBand"] = pd.cut(active["ChurnProb"], bins=[0, 0.25, 0.50, 1.0], labels=["Low", "Medium", "High"])

band = active.groupby("RiskBand", observed=True).agg(
    customers=("CustID", "count"),
    monthly_revenue=("MonthlyCharges", "sum"),
    avg_churn_prob=("ChurnProb", "mean"))
band["expected_monthly_loss"] = active.assign(x=active.MonthlyCharges * active.ChurnProb).groupby("RiskBand", observed=True)["x"].sum()
display(band)

print(f"Active customers: {len(active)}")
print(f"Monthly revenue of active customers: Rs {active.MonthlyCharges.sum():,.0f}")
print(f"Revenue at risk (High band): Rs {band.loc['High', 'monthly_revenue']:,.0f} per month")

**Business insight**
- There are **1,300 active customers paying ₹10.79 lakh a month**.
- **178 customers are in the High-risk band**. They pay **₹1.80 lakh a month** (about 17% of active revenue) and have an average churn probability of 54%.
- Across all bands, the **expected monthly revenue loss is about ₹3.27 lakh (about 30% of active revenue)** if nothing changes.
- The Medium band holds the largest expected loss (₹1.53 lakh), so retention should not stop at the High band.

In [ ]:
# Retention call list: active customers with the highest churn risk, sorted by revenue impact
watchlist = (active[active["RiskBand"] == "High"]
             .sort_values(["ChurnProb", "MonthlyCharges"], ascending=False)
             [["CustID", "Customer Name", "City", "Contract", "Internet", "Tenure",
               "MonthlyCharges", "ChurnProb"]]
             .rename(columns={"ChurnProb": "Churn probability"}))
display(watchlist.head(15))
# watchlist.to_excel("retention_watchlist.xlsx", index=False)   # uncomment to export

**Business insight**
- The High-risk list is made up almost entirely of **Month-to-month customers, mostly on Fiber, with a median tenure of 8 months**.
- This is an action-ready call list for the retention team. Uncomment the last line to export it to Excel.

## 7. What is the retention opportunity worth?
Month-to-month churn is the largest single source of lost revenue, so the scenarios below show the revenue saved if retention actions cut Month-to-month churn by 10%, 20% or 30%.

In [ ]:
# What-if: how much revenue can retention actions save?
base_churn = known["ChurnFlag"].mean()
mtm = known[known["Contract"] == "Month-to-month"]
mtm_lost_monthly = mtm.loc[mtm["ChurnFlag"] == 1, "MonthlyCharges"].sum()

scenarios = []
for reduction in [0.10, 0.20, 0.30]:          # relative reduction in Month-to-month churn
    saved_monthly = mtm_lost_monthly * reduction
    scenarios.append({"Reduction in Month-to-month churn": f"{reduction:.0%}",
                      "Customers saved": int(round(mtm["ChurnFlag"].sum() * reduction)),
                      "Monthly revenue saved (Rs)": saved_monthly,
                      "Annual revenue saved (Rs)": saved_monthly * 12})
pd.DataFrame(scenarios)

**Business insight**
- Cutting Month-to-month churn by **20%** would save about **103 customers, ₹0.87 lakh a month and ₹10.5 lakh a year**.
- Even a modest **10%** reduction is worth about **₹5.2 lakh a year**, which gives a budget ceiling for retention offers: any discount costing less than this per year pays for itself.

## 8. Executive summary

**Problem:** About one in three customers (33.8%) churns, costing roughly **₹5.67 lakh in monthly revenue (₹68 lakh a year)**.

**What drives churn (strongest to weakest, confirmed by statistical tests)**
1. **Contract type:** Month-to-month churns at 47.8% and produces about 78% of all churn. Two-year contracts churn at 10.9%.
2. **Tenure:** about half of all churn happens in the first 12 months. After two years, customers rarely leave.
3. **Internet service:** Fiber has the highest churn (37.7%) and about 63% of the lost revenue, mostly because Fiber customers are often on Month-to-month plans.
4. **Not significant:** age, city and gender.

**Predictive model:** ROC-AUC of about 0.75. It identifies **178 high-risk active customers (₹1.80 lakh a month)** and puts the expected loss from all active customers at about ₹3.27 lakh a month if nothing changes.

**Recommendations**
1. **Convert Month-to-month customers to 1- or 2-year plans** with discounts or benefits. A 20% cut in Month-to-month churn is worth about ₹10.5 lakh a year.
2. **Run the retention team from the risk-scored call list**, starting with the High band.
3. **Strengthen onboarding and check-ins in the first 6-12 months.**
4. **Review Fiber quality and pricing**, since it is the most valuable segment.
5. **Investigate the 2024-2025 signups,** whose churn is far above older cohorts.

**Limitations:** the model uses only four features, so it should be seen as a prioritisation tool, not a precise forecast. Unknown-status customers (35) are excluded. The data is a snapshot, so tenure and cohort effects are partly intertwined.